In [ ]:
import pandas as pd
import numpy as np
import blinksear as ear
import blinksdistance as distance
import blinkscolors as colors
import json
import matplotlib.pyplot as plt
from pathlib import Path

from blinksear import isBlinkByEar
from blinksdistance import BlinkDistance

In [ ]:
def generate_webgazer_dataframe(df):
    df1 = df[df["trial_type"] == "html-keyboard-response"]
    df_webgazer = df1[~df1["webgazer_data"].isna()]
    dfs = []
    for row in range(len(df_webgazer)):
        json_data = df_webgazer["webgazer_data"].iloc[row].replace("'", "\"")
        data_list = json.loads(json_data)
        df_tmp = pd.DataFrame(data_list)
        df_tmp["trial"] = row + 1
        dfs.append(df_tmp)
    return pd.concat(dfs)


In [ ]:
# This can load all data but is not storing df_webgazer in any place
files = Path("./data/all-exp-separados/").glob("*.csv")
dfs = []
i = 0
for file in files:
    if "blink" in str(file):
        print(file)
        df = pd.read_csv(file)
        df_tmp = generate_webgazer_dataframe(df)
        print(len(df_tmp))
        df_tmp["experiment"] = i
        df_tmp["file_name"] = str(file)
        i+=1
        dfs.append(df_tmp)

df_webgazer = pd.concat(dfs)
print(len(df_webgazer))

In [ ]:
blinkDistance = distance.BlinkDistance()


df_webgazer["isBlinkByEar"] = df_webgazer["importantKeypoints"].map(
    lambda x: isBlinkByEar(
        x["rightEyeTopArc"],
        x["rightEyeBottomArc"],
        x["leftEyeTopArc"],
        x["leftEyeBottomArc"],
    )
)
df_webgazer["isBlinkByDistance"] = df_webgazer["importantKeypoints"].map(
    lambda x: blinkDistance.isBlinkByDistance(
        x["rightEyeTopArc"],
        x["rightEyeBottomArc"],
        x["leftEyeTopArc"],
        x["leftEyeBottomArc"],
    )
)
# df_webgazer['isBlinkByEar'] = df_webgazer['importantKeypoints'].map(lambda x: isBlinkByEar(x['rightEyeTopArc'], x['rightEyeBottomArc'], x['leftEyeTopArc'], x['leftEyeBottomArc']))

df_webgazer

In [ ]:
df_webgazer.head()

In [ ]:
threshold_columns = [col for col in result.columns if col not in ['trial', 'file_name']]
result['should_be'] = result['trial'].apply(lambda x: 2 if x == 3 else 10)
result[result['trial']==3]


In [ ]:
 #resultados por participante blink ear
df_webgazer["Blink_sample_ear"] = df_webgazer["isBlinkByEar"].apply(lambda x: int(x[14]==True))
df_webgazer["Blink_sample_ear_diff"] = (df_webgazer['Blink_sample_ear'].shift(1)+1 == df_webgazer['Blink_sample_ear'])


result_per_participant_ear = df_webgazer.groupby(['file_name'])['Blink_sample_ear_diff'].sum().reset_index()
result_per_participant_ear

In [ ]:
 #resultados por participante blink distance
df_webgazer["Blink_sample_dist"] = df_webgazer["isBlinkByDistance"].apply(lambda x: int(x[6]==True))
df_webgazer["Blink_sample_dist_diff"] = (df_webgazer['Blink_sample_dist'].shift(1)+1 == df_webgazer['Blink_sample_dist'])


result_per_participant_dist = df_webgazer.groupby(['file_name'])['Blink_sample_dist_diff'].sum().reset_index()
result_per_participant_dist

In [ ]:
result_per_participant_ear
result_per_participant_dist


plt.scatter(list(range(25)), result_per_participant_ear['Blink_sample_ear_diff'], marker='o', color='blue', label='EAR')  # círculo
plt.scatter(list(range(25)), result_per_participant_dist['Blink_sample_dist_diff'], marker='s', color='red', label='Distance')   # cuadrado

plt.axhline(y=52, color="red", linestyle="--", lw=1, label= "Valor real")

plt.xlabel('Participante')
plt.ylabel('Cantidad de blinks')
plt.title('Comparacion entre metodos')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10)  # Leyenda fuera del gráfico
plt.grid(True)  # Activar la cuadrícula

# Mostrar
plt.show()


In [ ]:
 #resultados por participante por trial blink ear
df_webgazer["Blink_sample_ear"] = df_webgazer["isBlinkByEar"].apply(lambda x: int(x[14]==True))
df_webgazer["Blink_sample_ear_diff"] = (df_webgazer['Blink_sample_ear'].shift(1)+1 == df_webgazer['Blink_sample_ear'])


result_per_participant_ear = df_webgazer.groupby(['file_name', 'trial'])['Blink_sample_ear_diff'].sum().reset_index()
result_per_participant_ear


In [ ]:
 #resultados por participante por trial blink distance
df_webgazer["Blink_sample_dist"] = df_webgazer["isBlinkByDistance"].apply(lambda x: int(x[6]==True))
df_webgazer["Blink_sample_dist_diff"] = (df_webgazer['Blink_sample_dist'].shift(1)+1 == df_webgazer['Blink_sample_dist'])


result_per_participant_dist = df_webgazer.groupby(['file_name', 'trial'])['Blink_sample_dist_diff'].sum().reset_index()
result_per_participant_dist

In [ ]:
result_per_participant_ear
result_per_participant_dist

result_ear = result_per_participant_ear[result_per_participant_ear["Blink_sample_ear_diff"]>0]
result_dist = result_per_participant_dist[result_per_participant_dist["Blink_sample_dist_diff"]>0]

result_dist['error'] = np.where(
    result_dist['trial'] == 3,
    np.abs(result_dist['Blink_sample_dist_diff'] - 2),
    np.abs(result_dist['Blink_sample_dist_diff'] - 10)
)

result_ear['error'] = np.where(
    result_ear['trial'] == 3,
    np.abs(result_ear['Blink_sample_ear_diff'] - 2),
    np.abs(result_ear['Blink_sample_ear_diff'] - 10)
)

result_ear['metodo'] = 'ear'
result_dist['metodo'] = 'dist'

combined_df = pd.concat([result_ear, result_dist])

result_df = combined_df[['error', 'metodo']]
##print(result_df)

result_df.filter

agrupado = result_df.groupby('metodo')['error'].agg(['mean', 'std'])

print(agrupado)



In [ ]:

combined_df

In [ ]:

# Unimos los datos en un único DataFrame para facilitar el boxplot
data = pd.DataFrame({
    'EAR': result_per_participant_ear['Blink_sample_ear_diff'],
    'Distancia': result_per_participant_dist['Blink_sample_dist_diff'],
})

plt.rcParams['font.size'] = 14

# Generar boxplot
ax = data.boxplot(column=['EAR', 'Distancia'],
                  grid=True, figsize=(8,6))

plt.axhline(y=52, color="red", linestyle="--", lw=1, label= "Valor esperado")
plt.legend(loc='upper left', bbox_to_anchor=(0, 1), fontsize=12)  # Leyenda fuera del gráfico

plt.ylabel('Número de parpadeos detectados')
plt.title('Distribución de detecciones por método')
plt.show()


In [ ]:
#Boxplot con error en vez de cant de blinks
valor_esperado = 52

data = pd.DataFrame({
    'EAR': abs(result_per_participant_ear['Blink_sample_ear_diff'] - valor_esperado),
    'Distancia': abs(result_per_participant_dist['Blink_sample_dist_diff'] - valor_esperado),
})

plt.rcParams['font.size'] = 14

# Generar boxplot
ax = data.boxplot(column=['EAR', 'Distancia'],
                  grid=True, figsize=(8,6))

plt.ylabel('Error absoluto')
plt.title('Distribución de detecciones por método')
plt.show()